# PyTables to SQLite Converter

This notebook converts the `mitosis_train_patches.pytable` file into a SQLite database for efficient data loading in training pipelines.

## 1. Import Required Libraries

Import necessary libraries including tables (PyTables), sqlite3, pandas, and numpy for data handling and database operations.

In [1]:
import sqlite3
import tables
import numpy as np
import pandas as pd
from pathlib import Path
import logging
from typing import Optional, Dict, Any
import io

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Configuration
PYTABLE_PATH = "mitosis_train_patches.pytable"
SQLITE_PATH = "mitosis_train_patches.db"
BATCH_SIZE = 1000  # Number of rows to process at a time

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Load PyTables File

Open and load the mitosis_train_patches.pytable file using PyTables, exploring the file structure and available datasets.

In [2]:
# Open the PyTables file and explore its structure
h5_file = tables.open_file(PYTABLE_PATH, mode='r')

# Display file structure
print("PyTables File Structure:")
print("=" * 60)
for table in h5_file.walk_nodes("/", classname="Table"):
    print(f"  {table._v_pathname}")
print()

# Store reference to the main table (usually the root table)
node = h5_file.get_node("/")
logger.info(f"File opened: {PYTABLE_PATH}")
logger.info(f"Root node type: {type(node)}")

INFO:__main__:File opened: mitosis_train_patches.pytable
INFO:__main__:Root node type: <class 'tables.group.RootGroup'>


PyTables File Structure:



## 3. Inspect Data Structure

Examine the shape, dtype, and structure of the data in the PyTables file to understand what will be stored in SQLite.

In [3]:
# Explore the complete file structure
print("Complete file structure:")
print("=" * 60)
for node in h5_file.walk_nodes("/"):
    indent = "  " * (node._v_pathname.count('/') - 1)
    node_type = type(node).__name__
    if hasattr(node, 'shape'):
        print(f"{indent}{node._v_name} ({node_type}, shape={node.shape}, dtype={node.dtype})")
    else:
        print(f"{indent}{node._v_name} ({node_type})")
print()

# Find all arrays (EArray, Table, etc.) in the PyTables file
arrays_list = []
table_list = []

# Check for Tables first
for table in h5_file.walk_nodes("/", classname="Table"):
    table_list.append(table._v_pathname)

# If no tables, look for EArrays or other array types
if not table_list:
    for array in h5_file.walk_nodes("/", classname="EArray"):
        arrays_list.append(array._v_pathname)
    
    if arrays_list:
        print(f"Found {len(arrays_list)} EArray(s) - will combine into single table:")
        for arr_path in arrays_list:
            arr = h5_file.get_node(arr_path)
            print(f"  - {arr_path}: shape={arr.shape}, dtype={arr.dtype}")
        main_table_path = None
        use_earrays = True
    else:
        main_table_path = None
        use_earrays = False
else:
    print(f"Found {len(table_list)} Table(s):")
    for table_path in table_list:
        print(f"  - {table_path}")
    main_table_path = table_list[0]
    use_earrays = False

Complete file structure:
/ (RootGroup)
PHH3_label (EArray, shape=(np.int64(8948),), dtype=int8)
label_jc (EArray, shape=(np.int64(8948),), dtype=int8)
label_mdy (EArray, shape=(np.int64(8948),), dtype=int8)
mask (EArray, shape=(np.int64(8948), np.int64(64), np.int64(64)), dtype=uint8)
patch (EArray, shape=(np.int64(8948), np.int64(64), np.int64(64), np.int64(3)), dtype=uint8)
patch_id (EArray, shape=(np.int64(8948),), dtype=int32)
patch_id_unique (EArray, shape=(np.int64(8948),), dtype=int32)
scanner (EArray, shape=(np.int64(8948),), dtype=|S4)
slide_id (EArray, shape=(np.int64(8948),), dtype=int32)
tmp_label (EArray, shape=(np.int64(8948),), dtype=int64)

Found 10 EArray(s) - will combine into single table:
  - /PHH3_label: shape=(np.int64(8948),), dtype=int8
  - /label_jc: shape=(np.int64(8948),), dtype=int8
  - /label_mdy: shape=(np.int64(8948),), dtype=int8
  - /mask: shape=(np.int64(8948), np.int64(64), np.int64(64)), dtype=uint8
  - /patch: shape=(np.int64(8948), np.int64(64), np

## 4. Create SQLite Database Schema

Define and create the SQLite database schema with appropriate tables and columns to store patch data, including primary keys and data types.

In [4]:
# Helper function to map PyTables dtype to SQLite type
def pytables_to_sqlite_type(pytables_dtype: str, shape: tuple) -> str:
    """
    Map PyTables dtypes to SQLite type affinities.
    
    Parameters
    ----------
    pytables_dtype : str
        The PyTables data type string
    shape : tuple
        The shape of the array (determines if it's multidimensional)
    
    Returns
    -------
    str
        SQLite type affinity
    """
    # Multi-dimensional arrays stored as BLOB
    if len(shape) > 1:  
        return "BLOB"
    elif "int" in str(pytables_dtype):
        return "INTEGER"
    elif "float" in str(pytables_dtype):
        return "REAL"
    elif "bool" in str(pytables_dtype):
        return "INTEGER"
    else:
        return "TEXT"

# Create SQLite connection
conn = sqlite3.connect(SQLITE_PATH)
cursor = conn.cursor()

# Drop existing table if it exists
table_name = "mitosis_patches"
cursor.execute(f"DROP TABLE IF EXISTS {table_name}")

# Build CREATE TABLE statement based on data type
if use_earrays and arrays_list:
    # For EArrays: create table with one column per EArray
    create_sql = f"CREATE TABLE {table_name} (id INTEGER PRIMARY KEY AUTOINCREMENT, score REAL"
    
    for arr_path in arrays_list:
        arr = h5_file.get_node(arr_path)
        col_name = arr._v_name
        sql_type = pytables_to_sqlite_type(str(arr.dtype), arr.shape)
        create_sql += f", {col_name} {sql_type}"
    
    create_sql += ")"
    
    # Determine number of rows per array
    num_rows = h5_file.get_node(arrays_list[0]).shape[0]
else:
    # For Tables: create table from table structure
    table = h5_file.get_node(main_table_path)
    create_sql = f"CREATE TABLE {table_name} (id INTEGER PRIMARY KEY AUTOINCREMENT, score REAL"
    
    for col_name in table.colnames:
        col = table.col(col_name)
        sql_type = pytables_to_sqlite_type(str(col.dtype), col.shape)
        create_sql += f", {col_name} {sql_type}"
    
    create_sql += ")"
    num_rows = table.nrows

# Execute create table statement
cursor.execute(create_sql)
conn.commit()

# Create a partial index on score for only non-null positive scores
index_name = f"idx_{table_name}_score_positive"
index_sql = (
    f"CREATE INDEX IF NOT EXISTS {index_name} "
    f"ON {table_name}(score) "
    f"WHERE score IS NOT NULL AND score > 0"
)
cursor.execute(index_sql)
conn.commit()

logger.info(f"SQLite table created: {table_name}")
logger.info(f"Created partial index: {index_name}")
print(f"Table '{table_name}' created successfully!")
print(f"Created partial index: {index_name}")
print(f"Number of rows: {num_rows}")
print(f"SQL: {create_sql}")

INFO:__main__:SQLite table created: mitosis_patches
INFO:__main__:Created partial index: idx_mitosis_patches_score_positive


Table 'mitosis_patches' created successfully!
Created partial index: idx_mitosis_patches_score_positive
Number of rows: 8948
SQL: CREATE TABLE mitosis_patches (id INTEGER PRIMARY KEY AUTOINCREMENT, score REAL, PHH3_label INTEGER, label_jc INTEGER, label_mdy INTEGER, mask BLOB, patch BLOB, patch_id INTEGER, patch_id_unique INTEGER, scanner TEXT, slide_id INTEGER, tmp_label INTEGER)


In [5]:
# # Clean up old database before creating new one
# import os
# if os.path.exists(SQLITE_PATH):
#     os.remove(SQLITE_PATH)
#     print(f"✓ Removed old database: {SQLITE_PATH}")

## 5. Write Data to SQLite

Read data from the PyTables file in batches and insert it into the SQLite database efficiently, handling binary data and large arrays appropriately.

In [6]:
def serialize_array(arr: np.ndarray) -> bytes:
    """
    Serialize a numpy array to bytes for storage in SQLite BLOB.
    
    Parameters
    ----------
    arr : np.ndarray
        Array to serialize
        
    Returns
    -------
    bytes
        Serialized array
    """
    buffer = io.BytesIO()
    np.save(buffer, arr, allow_pickle=False)
    return buffer.getvalue()

def extract_value(value: np.ndarray) -> Any:
    """
    Extract native Python type from numpy value or array.
    Only serialize truly multidimensional (2D+) arrays as BLOB.
    
    Parameters
    ----------
    value : np.ndarray or scalar
        Value to process
        
    Returns
    -------
    Any
        Native Python type for scalars/1D arrays, BLOB for 2D+ arrays
    """
    # Handle numpy scalars
    if isinstance(value, np.generic):
        # For 0D arrays or numpy scalars
        if hasattr(value, 'item'):
            return value.item()  # Convert to native Python type
        else:
            return value
    
    # Handle arrays
    if isinstance(value, np.ndarray):
        # Check dimensionality after indexing
        if value.ndim == 0:
            # 0D array - extract scalar
            return value.item()
        elif value.ndim == 1:
            # 1D array - check if single element
            if len(value) == 1:
                return value[0].item() if hasattr(value[0], 'item') else value[0]
            else:
                # Multi-element 1D array - serialize as BLOB
                return serialize_array(value)
        else:
            # 2D+ array - serialize as BLOB
            return serialize_array(value)
    
    return value

print(f"Starting data migration: {num_rows} rows to convert")
print(f"Processing in batches of {BATCH_SIZE}")
print()

# Process data in batches
for batch_start in range(0, num_rows, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, num_rows)
    
    # Read batch data based on data type
    rows_to_insert = []
    
    if use_earrays and arrays_list:
        # For EArrays: read from each array and combine into rows
        for row_idx in range(batch_start, batch_end):
            row_values = []
            for arr_path in arrays_list:
                arr = h5_file.get_node(arr_path)
                value = arr[row_idx]
                
                # Extract value (scalar or BLOB for multidimensional)
                value = extract_value(value)
                
                row_values.append(value)
            rows_to_insert.append(row_values)
        
        col_names = [h5_file.get_node(arr_path)._v_name for arr_path in arrays_list]
    else:
        # For Tables: read from table
        table = h5_file.get_node(main_table_path)
        batch_data = table.read(start=batch_start, stop=batch_end)
        
        for row in batch_data:
            row_values = []
            for col_name in table.colnames:
                value = row[col_name]
                
                # Extract value (scalar or BLOB for multidimensional)
                value = extract_value(value)
                
                row_values.append(value)
            rows_to_insert.append(row_values)
        
        col_names = table.colnames
    
    # Insert batch with many execute calls compiled into one
    placeholders = ",".join(["?"] * len(col_names))
    insert_sql = f"INSERT INTO {table_name} ({','.join(col_names)}) VALUES ({placeholders})"
    
    cursor.executemany(insert_sql, rows_to_insert)
    conn.commit()
    
    # Progress reporting
    progress_pct = (batch_end / num_rows) * 100
    logger.info(f"Inserted rows {batch_start}-{batch_end} ({progress_pct:.1f}%)")
    print(f"Progress: {batch_end}/{num_rows} rows ({progress_pct:.1f}%)", end="\r")

print(f"\n✓ Data migration complete! {num_rows} rows inserted.")

Starting data migration: 8948 rows to convert
Processing in batches of 1000



INFO:__main__:Inserted rows 0-1000 (11.2%)


INFO:__main__:Inserted rows 1000-2000 (22.4%)


INFO:__main__:Inserted rows 2000-3000 (33.5%)


INFO:__main__:Inserted rows 3000-4000 (44.7%)


...

INFO:__main__:Inserted rows 5000-6000 (67.1%)


INFO:__main__:Inserted rows 6000-7000 (78.2%)


INFO:__main__:Inserted rows 7000-8000 (89.4%)


INFO:__main__:Inserted rows 8000-8948 (100.0%)


Progress: 8948/8948 rows (100.0%)
✓ Data migration complete! 8948 rows inserted.


## 6. Verify Database Contents

Query the SQLite database to confirm all data was written correctly, checking row counts and sampling records.

In [7]:
# Verification queries
cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
row_count = cursor.fetchone()[0]

print(f"Database Verification")
print("=" * 60)
print(f"Table name: {table_name}")
print(f"Total rows: {row_count}")
print(f"Expected rows: {num_rows}")
print(f"Match: {'✓ YES' if row_count == num_rows else '✗ NO'}")
print()

# Get schema info
cursor.execute(f"PRAGMA table_info({table_name})")
columns = cursor.fetchall()
print(f"Table schema ({len(columns)} columns):")
for col_info in columns:
    col_id, col_name, col_type, col_notnull, col_dflt, col_pk = col_info
    print(f"  {col_name}: {col_type} (pk={col_pk})")
print()

# Sample data from SQLite
print("Sample data (first 5 rows):")
cursor.execute(f"SELECT * FROM {table_name} LIMIT 5")
sample_rows = cursor.fetchall()
sample_df = pd.DataFrame(sample_rows, columns=[col[1] for col in columns])
display(sample_df)

# Get database file size
import os
db_size_mb = os.path.getsize(SQLITE_PATH) / (1024 * 1024)
print(f"\nDatabase file size: {db_size_mb:.2f} MB")

Database Verification
Table name: mitosis_patches
Total rows: 8948
Expected rows: 8948
Match: ✓ YES

Table schema (12 columns):
  id: INTEGER (pk=1)
  score: REAL (pk=0)
  PHH3_label: INTEGER (pk=0)
  label_jc: INTEGER (pk=0)
  label_mdy: INTEGER (pk=0)
  mask: BLOB (pk=0)
  patch: BLOB (pk=0)
  patch_id: INTEGER (pk=0)
  patch_id_unique: INTEGER (pk=0)
  scanner: TEXT (pk=0)
  slide_id: INTEGER (pk=0)
  tmp_label: INTEGER (pk=0)

Sample data (first 5 rows):


,id,score,PHH3_label,label_jc,label_mdy,mask,patch,patch_id,patch_id_unique,scanner,slide_id,tmp_label
0,1,None,0,-1,-1,"b""\x93NUMPY\x01\x00v\x00{'descr': '|u1', 'fort...","b""\x93NUMPY\x01\x00v\x00{'descr': '|u1', 'fort...",932,0,b'L',16,-1
1,2,None,0,-1,-1,"b""\x93NUMPY\x01\x00v\x00{'descr': '|u1', 'fort...","b""\x93NUMPY\x01\x00v\x00{'descr': '|u1', 'fort...",4823,1,b'L',16,-1
2,3,None,0,-1,-1,"b""\x93NUMPY\x01\x00v\x00{'descr': '|u1', 'fort...","b""\x93NUMPY\x01\x00v\x00{'descr': '|u1', 'fort...",5193,2,b'L',16,-1
3,4,None,0,-1,-1,"b""\x93NUMPY\x01\x00v\x00{'descr': '|u1', 'fort...","b'\x93NUMPY\x01\x00v\x00{\'descr\': \'|u1\', \...",5211,3,b'L',16,-1
4,5,None,0,-1,-1,"b""\x93NUMPY\x01\x00v\x00{'descr': '|u1', 'fort...","b""\x93NUMPY\x01\x00v\x00{'descr': '|u1', 'fort...",5332,4,b'L',16,-1



Database file size: 144.21 MB


## 7. Query and Test Database

Demonstrate how to query the SQLite database and prepare it for use in a PyTorch or custom dataloader with sample retrieval patterns.

In [8]:
def deserialize_array(data: bytes) -> np.ndarray:
    """
    Deserialize bytes back to numpy array.
    
    Parameters
    ----------
    data : bytes
        Serialized array data
        
    Returns
    -------
    np.ndarray
        Deserialized numpy array
    """
    try:
        buffer = io.BytesIO(data)
        return np.load(buffer, allow_pickle=False)
    except (ValueError, OSError):
        # If deserialization fails, return the raw bytes as an array
        return np.frombuffer(data, dtype=np.uint8)

class MitosisDataLoader:
    """
    Simple dataloader for SQLite-backed mitosis patch database.
    
    Attributes
    ----------
    db_path : str
        Path to SQLite database file
    table_name : str
        Name of the table to query
    """
    
    def __init__(self, db_path: str = SQLITE_PATH, table_name_input: str = "mitosis_patches"):
        self.db_path = db_path
        self.table_name = table_name_input
        self.conn = sqlite3.connect(db_path)
        self.cursor = self.conn.cursor()
        
        # Get total count
        self.cursor.execute(f"SELECT COUNT(*) FROM {self.table_name}")
        self.total_rows = self.cursor.fetchone()[0]
    
    def __len__(self) -> int:
        """Return total number of samples."""
        return self.total_rows
    
    def get_sample(self, idx: int) -> Dict[str, Any]:
        """
        Get a single sample by index.
        
        Parameters
        ----------
        idx : int
            Index of the sample (0-based)
            
        Returns
        -------
        dict
            Dictionary with column names as keys
        """
        query = f"SELECT * FROM {self.table_name} WHERE id = ?"
        self.cursor.execute(query, (idx + 1,))  # SQLite id is 1-based
        row = self.cursor.fetchone()
        
        if row is None:
            raise IndexError(f"Index {idx} out of range")
        
        # Get column names
        column_names = [description[0] for description in self.cursor.description]
        sample = dict(zip(column_names, row))
        
        return sample
    
    def get_batch(self, start_idx: int, batch_size: int) -> list:
        """
        Get a batch of samples.
        
        Parameters
        ----------
        start_idx : int
            Starting index
        batch_size : int
            Number of samples to retrieve
            
        Returns
        -------
        list of dict
            List of samples
        """
        query = f"SELECT * FROM {self.table_name} LIMIT ? OFFSET ?"
        self.cursor.execute(query, (batch_size, start_idx))
        rows = self.cursor.fetchall()
        
        column_names = [description[0] for description in self.cursor.description]
        return [dict(zip(column_names, row)) for row in rows]
    
    def close(self):
        """Close database connection."""
        self.conn.close()

# Example usage
print("Testing DataLoader:")
print("=" * 60)
loader = MitosisDataLoader()
print(f"Total samples in database: {len(loader)}")

# Get a single sample
sample = loader.get_sample(0)
print(f"\nSingle sample (index 0):")
for key, value in sample.items():
    if isinstance(value, bytes):
        try:
            arr = deserialize_array(value)
            print(f"  {key}: {type(arr).__name__} shape={arr.shape} dtype={arr.dtype}")
        except Exception as e:
            print(f"  {key}: bytes (raw, {len(value)} bytes)")
    else:
        if isinstance(value, (int, float, str)):
            print(f"  {key}: {value}")
        else:
            print(f"  {key}: {type(value).__name__}")

# Get a batch
batch = loader.get_batch(0, 3)
print(f"\nBatch of 3 samples:")
for i, sample in enumerate(batch):
    print(f"  Sample {i}: {len(sample)} fields")

loader.close()
print("\n✓ Database ready for use in training pipeline!")

Testing DataLoader:
Total samples in database: 8948

Single sample (index 0):
  id: 1
  score: NoneType
  PHH3_label: 0
  label_jc: -1
  label_mdy: -1
  mask: ndarray shape=(64, 64) dtype=uint8
  patch: ndarray shape=(64, 64, 3) dtype=uint8
  patch_id: 932
  patch_id_unique: 0
  scanner: ndarray shape=(1,) dtype=uint8
  slide_id: 16
  tmp_label: -1

Batch of 3 samples:
  Sample 0: 12 fields
  Sample 1: 12 fields
  Sample 2: 12 fields

✓ Database ready for use in training pipeline!


## Cleanup

Close database connections and PyTables file.

In [9]:
# Close connections
cursor.close()
conn.close()
h5_file.close()

logger.info(f"✓ Conversion complete! Database saved to: {SQLITE_PATH}")
print(f"✓ All connections closed")
print(f"✓ SQLite database ready at: {SQLITE_PATH}")
print(f"\nNext steps:")
print(f"  1. Use MitosisDataLoader class to load data")
print(f"  2. Integrate with PyTorch DataLoader for batch processing")
print(f"  3. Access data via get_sample() or get_batch() methods")

INFO:__main__:✓ Conversion complete! Database saved to: mitosis_train_patches.db


✓ All connections closed
✓ SQLite database ready at: mitosis_train_patches.db

Next steps:
  1. Use MitosisDataLoader class to load data
  2. Integrate with PyTorch DataLoader for batch processing
  3. Access data via get_sample() or get_batch() methods
